# Academic Advisor: model training (CSC 303)

This notebook trains the model behind the Academic Advisor web app. It runs end to end on
Google Colab (Runtime > Run all) and produces **`advisor_model.pkl`**, which you place in the
`models/` folder of the repo and serve with Flask.

**Pipeline:** synthetic student records -> exploratory look -> compare 3 models with
cross-validation -> evaluate the winner on a hold-out set -> save the `.pkl` bundle -> download.

> The data is synthetic (generated from plausible rules plus noise). It demonstrates the
> workflow; for real use, replace it with anonymised records from your department.

## 1. Setup

In [ ]:
import sys, sklearn, pandas as pd, numpy as np
print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

In [ ]:
!mkdir -p advisor models data
!touch advisor/__init__.py

## 2. Data generator
This cell writes the same `advisor/data.py` used by the web app.

In [ ]:
%%writefile advisor/data.py
"""Synthetic student-record generator and shared domain constants.

No real student records are used anywhere in this project. The generator encodes
common-sense relationships (attendance, study time, carry-overs, workload and sleep
all influence next-semester GPA) so that a model has something realistic to learn.
Swap `generate_dataset()` for a loader of real, anonymised records when available.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

# Features the ML model is trained on (order matters and is stored in the .pkl).
FEATURES = [
    "cgpa",
    "last_gpa",
    "attendance_pct",
    "study_hours_week",
    "assignment_pct",
    "carryovers",
    "credit_load",
    "sleep_hours",
    "extracurricular_hours",
]

TARGET = "next_standing"

# Ordered from worst to best.
CLASSES = ["At Risk", "Fair", "Good", "Excellent"]

# Nigerian 5-point scale (lower bound, label). Check your department's handbook.
DEGREE_CLASSES = [
    (4.50, "First Class"),
    (3.50, "Second Class Upper"),
    (2.40, "Second Class Lower"),
    (1.50, "Third Class"),
    (1.00, "Pass"),
]


def degree_class(cgpa: float) -> str:
    """Return the degree class for a CGPA on the 5-point scale."""
    for lower, label in DEGREE_CLASSES:
        if cgpa >= lower:
            return label
    return "Below Pass"


def gpa_to_standing(gpa: float) -> str:
    """Map a semester GPA to the standing label the model predicts."""
    if gpa >= 4.5:
        return "Excellent"
    if gpa >= 3.5:
        return "Good"
    if gpa >= 2.4:
        return "Fair"
    return "At Risk"


def generate_dataset(n: int = 4000, seed: int = 42) -> pd.DataFrame:
    """Create `n` synthetic student records with the next-semester standing label."""
    rng = np.random.default_rng(seed)

    ability = rng.normal(0.0, 1.0, n)  # hidden driver shared by several features

    cgpa = np.clip(3.4 + 0.75 * ability + rng.normal(0, 0.25, n), 1.0, 5.0)
    last_gpa = np.clip(cgpa + rng.normal(0, 0.45, n), 0.0, 5.0)
    attendance = np.clip(78 + 5 * ability + rng.normal(0, 11, n), 30, 100)
    study = np.clip(13 + 3 * ability + rng.normal(0, 5, n), 0, 40)
    assignment = np.clip(80 + 6 * ability + rng.normal(0, 10, n), 20, 100)
    carryovers = np.clip(rng.poisson(np.clip(1.0 - 0.8 * ability, 0.05, None)), 0, 6)
    credit_load = rng.choice([15, 18, 21, 24, 27], size=n, p=[0.10, 0.30, 0.30, 0.22, 0.08])
    sleep = np.clip(rng.normal(6.8, 1.2, n), 3, 10)
    extracurricular = np.clip(rng.gamma(2.0, 3.0, n), 0, 25)

    # Hidden "true" process for next-semester GPA, plus noise.
    next_gpa = (
        0.35
        + 0.60 * cgpa
        + 0.25 * last_gpa
        + 0.012 * (attendance - 75)
        + 0.020 * (study - 13)
        + 0.008 * (assignment - 80)
        - 0.12 * carryovers
        - 0.040 * np.maximum(credit_load - 21, 0)
        + 0.08 * np.clip(sleep - 6.0, -3, 2)
        - 0.025 * np.maximum(extracurricular - 10, 0)
        + rng.normal(0, 0.30, n)
    )
    next_gpa = np.clip(next_gpa, 0.0, 5.0)

    df = pd.DataFrame(
        {
            "cgpa": cgpa.round(2),
            "last_gpa": last_gpa.round(2),
            "attendance_pct": attendance.round(0),
            "study_hours_week": study.round(1),
            "assignment_pct": assignment.round(0),
            "carryovers": carryovers.astype(int),
            "credit_load": credit_load.astype(int),
            "sleep_hours": sleep.round(1),
            "extracurricular_hours": extracurricular.round(1),
        }
    )
    df[TARGET] = [gpa_to_standing(g) for g in next_gpa]
    return df


if __name__ == "__main__":
    out = generate_dataset()
    out.to_csv("data/students.csv", index=False)
    print(out[TARGET].value_counts())
    print(f"Wrote {len(out)} rows to data/students.csv")

## 3. Generate and inspect the dataset

In [ ]:
import matplotlib.pyplot as plt
from advisor.data import FEATURES, TARGET, CLASSES, generate_dataset

df = generate_dataset(n=4000, seed=42)
df.to_csv("data/students.csv", index=False)
print(df.shape)
df.head()

In [ ]:
print(df[TARGET].value_counts().reindex(CLASSES))
df.describe().round(2).T[["mean", "std", "min", "max"]]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df[TARGET].value_counts().reindex(CLASSES).plot.bar(ax=ax[0], color="#2a4bd7")
ax[0].set_title("Next-semester standing (target)")
ax[0].set_xlabel(""); ax[0].tick_params(axis="x", rotation=0)

corr = df[FEATURES].corr()
im = ax[1].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax[1].set_xticks(range(len(FEATURES)), FEATURES, rotation=90)
ax[1].set_yticks(range(len(FEATURES)), FEATURES)
ax[1].set_title("Feature correlations")
fig.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.show()

## 4. Compare candidate models (5-fold cross-validation)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

def rf():  # must match advisor/model.py build_model()
    return RandomForestClassifier(n_estimators=200, min_samples_leaf=8,
                                  class_weight="balanced", random_state=42, n_jobs=-1)

candidates = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
    "Random Forest": rf(),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, model in candidates.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro")
    rows.append({"model": name, "macro_f1_mean": scores.mean().round(4), "macro_f1_std": scores.std().round(4)})
pd.DataFrame(rows).sort_values("macro_f1_mean", ascending=False)

The scores are close, so the **Random Forest** is used: it handles non-linear interactions, needs no
feature scaling, exposes feature importances for the app, and stays small once compressed.

## 5. Train the final model and evaluate on the hold-out set

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score

model = rf().fit(X_train, y_train)
pred = model.predict(X_test)
print(classification_report(y_test, pred, labels=CLASSES))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=CLASSES, ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title("Confusion matrix (hold-out)")
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
imp.plot.barh(ax=ax[1], color="#2a4bd7"); ax[1].set_title("Feature importance")
plt.tight_layout(); plt.show()

## 6. Save the `.pkl` bundle and verify it reloads

In [ ]:
import joblib

bundle = {
    "model": model,
    "feature_names": list(FEATURES),
    "classes": [str(c) for c in model.classes_],
    "metrics": {
        "accuracy": round(float(accuracy_score(y_test, pred)), 4),
        "macro_f1": round(float(f1_score(y_test, pred, average="macro")), 4),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
    },
    "sklearn_version": sklearn.__version__,
}
joblib.dump(bundle, "models/advisor_model.pkl", compress=3)

# Sanity check: reload from disk and predict for one hypothetical student.
loaded = joblib.load("models/advisor_model.pkl")
student = pd.DataFrame([{
    "cgpa": 3.2, "last_gpa": 2.9, "attendance_pct": 70, "study_hours_week": 8,
    "assignment_pct": 75, "carryovers": 1, "credit_load": 24, "sleep_hours": 6, "extracurricular_hours": 10,
}])[loaded["feature_names"]]
print({c: round(float(p), 3) for c, p in zip(loaded["classes"], loaded["model"].predict_proba(student)[0])})
print("Saved models/advisor_model.pkl", loaded["metrics"])

## 7. Download the model

In [ ]:
try:
    from google.colab import files
    files.download("models/advisor_model.pkl")
except ImportError:
    print("Not running on Colab; the file is at models/advisor_model.pkl")

**Next:** copy the downloaded `advisor_model.pkl` into the repo's `models/` folder, run
`python app.py`, and test it in the browser. If loading fails with a scikit-learn version error,
install the version printed in section 1 locally (`pip install scikit-learn==<version>`).